# Pandas Fundamentals

:::{admonition} Working through this notebook
:class: tip
This page is a Jupyter notebook. **Download it** using the ⬇ button in the top-right (or copy-paste the cells into a fresh notebook), open it in your environment (JupyterLab on LEAP or Colab), and step through the cells. When you reach a **Try it** admonition, experiment in your own cells before moving on.
:::

:::{admonition} In-class assignment — 10 points
:class: note
The **"Try it"** exercises in this notebook are part of your in-class assignment for this section. Complete them in your own copy of the notebook, push it to your week folder, and post the notebook link on the matching **Courseworks** assignment. (One 10-point assignment covers all the lecture notebooks in this section.)
:::

Let's start by importing the libraries we'll use throughout this lecture: **pandas** (conventionally imported as `pd`), NumPy, and matplotlib's `pyplot`.

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
%matplotlib inline

## Series

Pandas gives you two main objects to work with: a **Series** (one-dimensional, like a single column of labelled data) and a **DataFrame** (two-dimensional, like a table or spreadsheet). DataFrames are built up from Series — each column of a DataFrame is itself a Series — so we'll start with Series and then build up to DataFrame.

A Series is a one-dimensional array of data with an _index_: labels we use to access each value.

There are many ways to [create a Series](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.html). Let's start with a simple example using planet names and masses.

To create a Series, we pass two lists to `pd.Series(...)`: the data **values**, and an `index` containing the **labels** for those values. The output below will show the labels on the left and the values on the right:

In [ ]:
names = ['Mercury', 'Venus', 'Earth']
values = [0.3e24, 4.87e24, 5.97e24]
masses = pd.Series(values, index=names)
masses

Series have built in plotting methods.

In [ ]:
masses.plot(kind='bar')

Arithmetic operations and most numpy function can be applied to Series.
An important point is that the Series keep their index during such operations.

In [ ]:
np.log(masses) / masses**2

We can access the underlying index object if we need to:

In [ ]:
masses.index

### Indexing

We can get values back out using the index via the `.loc` attribute

In [ ]:
masses.loc['Earth']

Or by raw position using `.iloc`

In [ ]:
masses.iloc[2]

We can pass a list or array to loc to get multiple rows back:

In [ ]:
masses.loc[['Venus', 'Earth']]

And we can even use slice notation

In [ ]:
masses.loc['Mercury':'Earth']

In [ ]:
masses.iloc[:2]

If we need to, we can always get the raw data back out as well

In [ ]:
masses.values # a numpy array

In [ ]:
masses.index # a pandas Index object

:::{admonition} Try it
:class: tip
In a fresh code cell, build your own `Series` from a Python list of values and a list of labels (any topic of your choice — e.g., capital cities mapped to populations). Use `.loc[]` to access one element by label, and `.iloc[]` to access the same element by position. Then slice with `.loc[]` over a *label* range — note that both the start and end labels are included (unlike Python list slicing).
:::

## DataFrame

There is a lot more to Series, but they are limited to a single "column". A more useful Pandas data structure is the DataFrame. A DataFrame is basically a bunch of series that share the same index. It's a lot like a table in a spreadsheet.

Below we create a DataFrame.

Now let's build a `DataFrame`. The most common pattern is to start with a Python dictionary mapping column names to lists of values, then pass it to `pd.DataFrame`:

In [ ]:
data = {'mass': [0.3e24, 4.87e24, 5.97e24],       # kg
        'diameter': [4879e3, 12_104e3, 12_756e3], # m
        'rotation_period': [1407.6, np.nan, 23.9] # h
       }
df = pd.DataFrame(data, index=['Mercury', 'Venus', 'Earth'])
df

Pandas handles missing data very elegantly, keeping track of it through all calculations.

DataFrames come with a lot of built-in inspection and summary methods. A few of the most useful:

In [ ]:
df.info()

A wide range of statistical functions are available on both Series and DataFrames.

In [ ]:
df.min()

In [ ]:
df.mean()

In [ ]:
df.std()

In [ ]:
df.describe()

We can get a single column as a Series using python's getitem syntax on the DataFrame object.

In [ ]:
df['mass']

...or using attribute syntax.

In [ ]:
df.mass

:::{admonition} Try it
:class: tip
Build your own `DataFrame` from a dictionary of columns (any topic — e.g. a few cities with `population`, `area`, and `elevation`), passing an `index` of labels. Include at least one `np.nan` so you can see how pandas reports missing data. Inspect its structure with `.info()` and `.describe()`, then compute a few summary statistics with `.mean()`, `.min()`, and `.std()` — try them on the whole DataFrame and on a single column (e.g. `df['population'].mean()`). Finally, pull a column out as a Series using both `df['col']` and `df.col`.
:::

Indexing works very similar to series

In [ ]:
df.loc['Earth']

In [ ]:
df.iloc[2]

But we can also specify the column we want to access

In [ ]:
df.loc['Earth', 'mass']

In [ ]:
df.iloc[:2, 0]

If we make a calculation using columns from the DataFrame, it will keep the same index:

In [ ]:
volume =  4/3 * np.pi * (df.diameter/2)**3
df.mass / volume

Which we can easily add as another column to the DataFrame:

In [ ]:
df['density'] = df.mass / volume
df

## Merging Data

Pandas supports a wide range of methods for merging different datasets. These are described extensively in the [documentation](https://pandas.pydata.org/pandas-docs/stable/merging.html). Here we just give a few examples.

Pandas can merge DataFrames and Series intelligently along their shared index. Let's create a separate `Series` of planet temperatures and combine it with our existing DataFrame:

In [ ]:
temperature = pd.Series([167, 464, 15, -65],
                     index=['Mercury', 'Venus', 'Earth', 'Mars'],
                     name='temperature')
temperature

In [ ]:
df.join(temperature)

By default, `join` keeps the calling DataFrame's index — a *left* join — so Mars is dropped, since it isn't one of our three planets. The `how` keyword changes this: `how='right'` keeps the *other* object's index instead, so Mars appears (with `NaN` for the columns we don't have for it):

In [ ]:
df.join(temperature, how='right')

`reindex` is a more general tool: it conforms a DataFrame to a brand-new index that you supply, inserting rows of `NaN` wherever a label wasn't already present:

In [ ]:
everyone = df.reindex(['Mercury', 'Venus', 'Earth', 'Mars'])
everyone

:::{admonition} Try it
:class: tip
Create a new `Series` of values indexed by some of the planets — you can include one that isn't in the DataFrame (e.g. Mars or Jupiter) — and `join` it onto the planets DataFrame. Try `how='right'` and compare the result with the default join. Then call `.reindex(...)` with your own list of planet names and notice how pandas fills in `NaN` for labels that weren't present.
:::

Another powerful way to select data is to pick rows based on a **condition** rather than by their label or position. A comparison like `df.mass > 4e24` is checked for every row and returns a *boolean Series* — a column of `True`/`False` values, one per planet. Placing that inside `df[...]` keeps only the rows where the value is `True`. For example, to get just the planets more massive than `4e24` kg:

In [ ]:
adults = df[df.mass > 4e24]
adults

Since that condition is itself just a Series of `True`/`False` values, we can also store it as a new column — a handy way to flag the rows that meet the criterion:

In [ ]:
df['is_big'] = df.mass > 4e24
df

### Modifying Values

We often want to modify values in a dataframe based on some rule. To modify values, we need to use `.loc` or `.iloc`

In [ ]:
df.loc['Earth', 'mass'] = 5.98e+24
df.loc['Venus', 'diameter'] += 1
df

:::{admonition} Try it
:class: tip
Take the planets DataFrame from above. Use `.loc[]` to get all of Earth's data, then `.iloc[:3]` to get the first three rows. Add a new column called `density_check` computed as `df.mass / (4/3 * np.pi * (df.diameter/2)**3)`, then filter the DataFrame to planets with mass greater than 1e24.
:::

## Plotting

DataFrames have all kinds of [useful plotting](https://pandas.pydata.org/pandas-docs/stable/visualization.html) built in.

In [ ]:
df.plot(kind='scatter', x='mass', y='diameter', grid=True)

In [ ]:
df.plot(kind='bar')

:::{admonition} Try it
:class: tip
Make a scatter plot of `diameter` vs `mass` from the planets DataFrame. Then make a bar chart of each planet's mass. (You can stay with `df.plot(kind='...')`, or use `df.plot.scatter(...)` / `df.plot.bar(...)` — both work.)
:::

## Time Indexes

Indexes are very powerful. They are a big part of why Pandas is so useful. There are different indices for different types of data. Time Indexes are especially great!

To build a time-indexed Series we first need a sequence of dates. `pd.date_range` generates one for us: we give it a `start` and `end` date and a frequency `freq` (here `'D'` for one timestamp per day), and it returns a `DatetimeIndex` of evenly spaced timestamps. Below we make two years of daily dates, then use them as the index of a `Series` whose values trace a seasonal sine wave — `.dayofyear` counts 1–365 through the year, so one full cycle spans a year:

In [ ]:
two_years = pd.date_range(start='2014-01-01', end='2016-01-01', freq='D')
timeseries = pd.Series(np.sin(2 *np.pi *two_years.dayofyear / 365),
                       index=two_years)
timeseries.plot()

We can use python's slicing notation inside `.loc` to select a date range.

In [ ]:
timeseries.loc['2015-01-01':'2015-07-01'].plot()

The TimeIndex object has lots of useful attributes

In [ ]:
timeseries.index.month

In [ ]:
timeseries.index.day

:::{admonition} Try it
:class: tip
Use `pd.date_range(start='2023-01-01', end='2023-12-31', freq='D')` to create a year of daily timestamps. Wrap them in a `Series` whose values are `np.random.randn(len(...))`. Slice it to get just March through May 2023 (use `.loc['2023-03':'2023-05']`), then call `.plot()`.
:::

## Reading Data Files: Weather Station Data

In this example, we will use NOAA weather station data from https://www.ncdc.noaa.gov/data-access/land-based-station-data.

The details of files we are going to read are described in this README file (<ftp://ftp.ncdc.noaa.gov/pub/data/uscrn/products/daily01/README.txt>).

In [ ]:
import pooch
POOCH = pooch.create(
    path=pooch.os_cache("noaa-data"),
    base_url="doi:10.5281/zenodo.5564850/",
    registry={
        "data.txt": "md5:5129dcfd19300eb8d4d8d1673fcfbcb4",
    },
)
datafile = POOCH.fetch("data.txt")
datafile

In [ ]:
! head {datafile}

We now have a text file on our hard drive called `data.txt`. Examine it.

To read it into pandas, we will use the [read_csv](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.read_csv.html) function. This function is incredibly complex and powerful. You can use it to extract data from almost any text file. However, you need to understand how to use its various options.

With no options, this is what we get.

In [ ]:
df = pd.read_csv(datafile)
df.head()

Pandas failed to identify the different columns. This is because it expected a standard CSV (comma-separated values) file, but in our file the values are separated by whitespace instead — and not by a single space: the amount of whitespace between columns varies. We tell pandas how the columns are separated with the `sep` keyword.

The value `r'\s+'` is a small *regular expression* — a pattern for matching text. Here `\s` stands for any whitespace character (a space or a tab) and `+` means "one or more in a row," so together they match any run of whitespace, however wide. The leading `r` makes it a *raw string*, which tells Python to leave the backslash alone instead of treating it as a special character.

In [ ]:
df = pd.read_csv(datafile, sep=r'\s+')
df.head()

Great! It worked. 

If we look closely, we will see there are lots of -99 and -9999 values in the file. The README file (<ftp://ftp.ncdc.noaa.gov/pub/data/uscrn/products/daily01/README.txt>) tells us that these are values used to represent missing data. Let's tell this to pandas.

In [ ]:
df = pd.read_csv(datafile, sep=r'\s+', na_values=[-9999.0, -99.0])
df.head()

Great. The missing data is now represented by `NaN`.

What data types did pandas infer?

In [ ]:
df.info()

One problem here is that pandas did not recognize the `LST_DATE` column as a date. Let's help it.

In [ ]:
df = pd.read_csv(datafile, sep=r'\s+',
                 na_values=[-9999.0, -99.0],
                 parse_dates=[1])
df.info()

It worked! Finally, let's tell pandas to use the date column as the index.

In [ ]:
df = df.set_index('LST_DATE')
df.head()

We can now access values by time:

In [ ]:
df.loc['2017-08-07']

Or use slicing to get a range:

In [ ]:
df.loc['2017-07-01':'2017-07-31']

### Quick Statistics

`describe()` is a fast way to get a feel for a freshly loaded dataset: in one call it reports the count, mean, standard deviation, minimum, the 25/50/75% percentiles, and maximum for every numeric column. It's a good first sanity check — unrealistic values or columns full of `NaN` often jump out right away.

In [ ]:
df.describe()

### Plotting Values

We can now quickly make plots of the data

In [ ]:
fig, ax = plt.subplots(ncols=2, nrows=2, figsize=(14,14))

df.iloc[:, 4:8].boxplot(ax=ax[0,0])
df.iloc[:, 10:14].boxplot(ax=ax[0,1])
df.iloc[:, 14:17].boxplot(ax=ax[1,0])
df.iloc[:, 18:22].boxplot(ax=ax[1,1])


ax[1, 1].set_xticklabels(ax[1, 1].get_xticklabels(), rotation=90);

Pandas is very "time aware":

In [ ]:
df.T_DAILY_MEAN.plot()

Note: we could also manually create an axis and plot into it.

In [ ]:
fig, ax = plt.subplots()
df.T_DAILY_MEAN.plot(ax=ax)
ax.set_title('Pandas Made This!')

In [ ]:
df[['T_DAILY_MIN', 'T_DAILY_MEAN', 'T_DAILY_MAX']].plot()

:::{admonition} Try it
:class: tip
Slice the NOAA `df` to a single month of your choice (e.g., `df.loc['2018-06']`). Call `.describe()` on that subset to get summary stats. Then plot `T_DAILY_MEAN` against the index for that month.
:::

### Resampling

Because pandas understands time, it can **resample** a time series — change how often the data is reported by grouping together all the timestamps that fall in each new time period. Our data is daily, but we might want monthly values; resampling to a monthly frequency gathers all the days in each month so we can summarize them (mean, max, …). It's much like `groupby`, except the groups are consecutive time intervals. The target frequency is given by a short code — below, `'MS'` means "month start" (one value per month); other common codes are `'D'` (daily), `'W'` (weekly), and `'YE'` (year end).

Calling `.resample(...)` returns a *resampler object* — a grouping by time frequency. Apply an aggregation (like `.mean()`) to actually get a smaller DataFrame:

In [ ]:
rs_obj = df.select_dtypes(include='number').resample('MS')
rs_obj

In [ ]:
rs_obj.mean()

We can chain all of that together

In [ ]:
df_mm = df.select_dtypes(include='number').resample('MS').mean()
df_mm[['T_DAILY_MIN', 'T_DAILY_MEAN', 'T_DAILY_MAX']].plot()

:::{admonition} Try it
:class: tip
Use `df.resample('YE').mean()` to get yearly means and plot `T_DAILY_MAX` against the resulting index as a line chart. Then do the same with `'10D'` (10-day bins) — the lower-frequency series should look smoother.
:::

## Recap

You've now met the two core pandas objects and the everyday ways to work with them:

- **Series and DataFrames** — labelled 1-D and 2-D data, with `.info()` and `.describe()` for a quick look.
- **Indexing** — `.loc` (by label), `.iloc` (by position), and boolean masks to filter rows by a condition.
- **Building and combining** — adding computed columns, and joining a Series or DataFrame on a shared index.
- **Reading real data** — `read_csv` with `sep`, `na_values`, and `parse_dates`, then setting a column as the index.
- **Time series** — a `DatetimeIndex`, slicing by date, plotting straight from a DataFrame, and a first taste of `resample`.

Next we go deeper into grouping operations — `groupby`, `resample`, and `rolling` — in [Pandas: Groupby](./pandas_groupby.ipynb).